# Project 95 — Local Debugging Workflow Agent
## Error → Hypothesize → Investigate → Fix

**Stack:** LangChain · Ollama · Pydantic · LangGraph · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langgraph pydantic

## Step 1 — Define Bug Reports

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import TypedDict, Optional
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

bug_reports = [
    {
        "id": "BUG-001",
        "title": "Users get 500 error on login",
        "description": "Since deploy v2.3.1, random users get HTTP 500 on login. "
                       "Happens ~10% of requests. Error log shows: "
                       "'ConnectionPool: max retries exceeded with url: /auth/token'. "
                       "Started after we increased session timeout from 30min to 2hrs.",
        "stack_trace": "requests.exceptions.ConnectionError: HTTPSConnectionPool(host='auth-svc', port=443): "
                       "Max retries exceeded",
        "env": "production, k8s, 3 replicas",
    },
    {
        "id": "BUG-002",
        "title": "Dashboard data shows yesterday's numbers",
        "description": "The analytics dashboard consistently shows data from the previous day. "
                       "Cache was recently moved from Redis to memcached. "
                       "TTL is set to 3600 seconds.",
        "stack_trace": "",
        "env": "production, AWS",
    },
    {
        "id": "BUG-003",
        "title": "Memory leak in background job processor",
        "description": "Worker pods OOM crash every 6 hours. Memory usage grows linearly. "
                       "Processing ~1000 jobs/hour. Each job downloads a file, processes it, "
                       "and uploads the result. File sizes range 1MB-50MB.",
        "stack_trace": "signal: killed (OOMKilled), last memory: 3.8Gi (limit: 4Gi)",
        "env": "production, k8s, 4Gi memory limit",
    },
]
print(f"Bug reports: {len(bug_reports)}")

## Step 2 — Hypothesis Generation

In [ ]:
class Hypothesis(BaseModel):
    description: str
    probability: float = Field(ge=0, le=1)
    evidence_for: list[str]
    evidence_against: list[str]
    investigation_steps: list[str]

class BugAnalysis(BaseModel):
    bug_id: str
    severity: str = Field(description="P0, P1, P2, P3")
    category: str = Field(description="infrastructure, logic, data, config, resource")
    hypotheses: list[Hypothesis]
    most_likely: str
    immediate_mitigation: str

analyzer = llm.with_structured_output(BugAnalysis)

analyses = []
for bug in bug_reports:
    analysis = analyzer.invoke(
        f"Analyze this bug and generate hypotheses:\n\n"
        f"Title: {bug['title']}\n"
        f"Description: {bug['description']}\n"
        f"Stack trace: {bug['stack_trace']}\n"
        f"Environment: {bug['env']}"
    )
    analyses.append(analysis)
    print(f"\n{analysis.bug_id} [{analysis.severity}] {analysis.category}")
    print(f"  Most likely: {analysis.most_likely}")
    print(f"  Hypotheses: {len(analysis.hypotheses)}")
    for h in analysis.hypotheses[:3]:
        print(f"    • {h.description[:60]}... (P={h.probability:.0%})")
    print(f"  Mitigation: {analysis.immediate_mitigation[:80]}")

## Step 3 — Debug Investigation Flow (LangGraph)

In [ ]:
from langgraph.graph import StateGraph, END

class DebugState(TypedDict):
    bug: dict
    analysis: dict
    investigation_log: list[str]
    root_cause: str
    fix_plan: str
    status: str

def investigate(state: DebugState) -> DebugState:
    chain = ChatPromptTemplate.from_messages([
        ("system", "You are investigating a bug. Simulate running the investigation steps "
         "and report findings. Be specific about what each step revealed."),
        ("human", "Bug: {bug}\nHypotheses: {hypotheses}\nInvestigate:")
    ]) | llm | StrOutputParser()
    findings = chain.invoke({
        "bug": json.dumps(state["bug"]),
        "hypotheses": json.dumps(state["analysis"]),
    })
    state["investigation_log"].append(findings)
    return state

def diagnose(state: DebugState) -> DebugState:
    chain = ChatPromptTemplate.from_messages([
        ("system", "Based on investigation, determine the root cause. Be specific."),
        ("human", "Bug: {bug}\nFindings: {findings}\nRoot cause:")
    ]) | llm | StrOutputParser()
    state["root_cause"] = chain.invoke({
        "bug": json.dumps(state["bug"]),
        "findings": state["investigation_log"][-1],
    })
    return state

def plan_fix(state: DebugState) -> DebugState:
    chain = ChatPromptTemplate.from_messages([
        ("system", "Create a detailed fix plan with code changes, config updates, and verification steps."),
        ("human", "Root cause: {cause}\nEnvironment: {env}\nFix plan:")
    ]) | llm | StrOutputParser()
    state["fix_plan"] = chain.invoke({
        "cause": state["root_cause"],
        "env": state["bug"].get("env", "unknown"),
    })
    state["status"] = "fix_ready"
    return state

graph = StateGraph(DebugState)
graph.add_node("investigate", investigate)
graph.add_node("diagnose", diagnose)
graph.add_node("plan_fix", plan_fix)
graph.set_entry_point("investigate")
graph.add_edge("investigate", "diagnose")
graph.add_edge("diagnose", "plan_fix")
graph.add_edge("plan_fix", END)
workflow = graph.compile()

print("Debug workflow: investigate → diagnose → plan_fix")

## Step 4 — Run Debugging for All Bugs

In [ ]:
debug_results = []
for bug, analysis in zip(bug_reports, analyses):
    result = workflow.invoke({
        "bug": bug,
        "analysis": analysis.model_dump(),
        "investigation_log": [],
        "root_cause": "",
        "fix_plan": "",
        "status": "investigating",
    })
    debug_results.append(result)
    print(f"\n{'='*50}")
    print(f"{bug['id']}: {bug['title']}")
    print(f"Root cause: {result['root_cause'][:150]}")
    print(f"\nFix plan: {result['fix_plan'][:200]}...")

## What You Learned
- **Hypothesis-driven debugging** with structured analysis
- **LangGraph debug workflow** (investigate → diagnose → fix)
- **Root cause analysis** from error traces
- **Automated fix planning** with verification steps